# MagicBrush Pro - Fast GPU Backend Server
Run this notebook on **Google Colab (T4 GPU)** to instantly load the diffusion models and provide an API URL for your local React frontend.

**Instructions:**
1. Go to **Runtime -> Change runtime type** and ensure **Hardware accelerator** is set to **T4 GPU**.
2. Make sure you have an Ngrok account (free). Go to `https://dashboard.ngrok.com/get-started/your-authtoken` and paste your token in the cell below.
3. Run all cells below.
4. Copy the **Public string** printed at the very bottom cell.
5. Paste that string into your local `magicbrush-pro/frontend/app.js` file (replace `http://localhost:8000`).
6. Save `app.js` and refresh your frontend `http://localhost:3001`.

In [ ]:
!pip install fastapi python-multipart uvicorn torch torchvision diffusers transformers accelerate lpips Pillow numpy opencv-python-headless pyngrok nest-asyncio

In [ ]:
utils_code = """import base64
import io
import cv2
import numpy as np
from PIL import Image

def decode_base64_image(base64_str: str) -> Image.Image:
    if "," in base64_str:
        base64_str = base64_str.split(",")[1]
    image_data = base64.b64decode(base64_str)
    image = Image.open(io.BytesIO(image_data))
    return image.convert("RGB")

def encode_image_base64(image: Image.Image) -> str:
    buffered = io.BytesIO()
    image.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return "data:image/png;base64," + img_str

def preprocess_image(image: Image.Image):
    image = image.resize((512, 512), Image.Resampling.LANCZOS)
    return image
"""

with open('utils.py', 'w') as f:
    f.write(utils_code)

print('utils.py created.')

In [ ]:
scoring_code = """import torch
import lpips
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torchvision.transforms as transforms

class Scorer:
    def __init__(self, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        print(f"Loading CLIP and LPIPS models on {device}...")
        
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
        
        self.lpips_model = lpips.LPIPS(net='vgg').to(self.device)
        
        self.to_tensor = transforms.ToTensor()

    def score(self, original_image: Image.Image, generated_image: Image.Image, prompt: str):
        with torch.no_grad():
            inputs = self.clip_processor(text=[prompt], images=generated_image, return_tensors="pt", padding=True).to(self.device)
            outputs = self.clip_model(**inputs)
            clip_score = outputs.logits_per_image[0][0].item()

            img0 = self.to_tensor(original_image).unsqueeze(0).to(self.device) * 2 - 1 
            img1 = self.to_tensor(generated_image).unsqueeze(0).to(self.device) * 2 - 1 
            lpips_score = self.lpips_model(img0, img1).item()

        final_score = (0.7 * clip_score) - (0.3 * lpips_score)
        
        return {
            "clip_score": round(clip_score, 2),
            "lpips_score": round(lpips_score, 4),
            "final_score": round(final_score, 2)
        }

_scorer = None
def get_scorer():
    global _scorer
    if _scorer is None:
        _scorer = Scorer()
    return _scorer
"""

with open('scoring.py', 'w') as f:
    f.write(scoring_code)

print('scoring.py created.')

In [ ]:
model_pipeline_code = """import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image

class InpaintPipeline:
    def __init__(self, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        print(f"Loading SD Inpainting model on {device}...")

        dtype = torch.float16 if device == "cuda" else torch.float32

        # runwayml/stable-diffusion-inpainting is the canonical inpainting checkpoint
        self.pipe = StableDiffusionInpaintPipeline.from_pretrained(
            "runwayml/stable-diffusion-inpainting",
            torch_dtype=dtype,
            safety_checker=None,
        ).to(self.device)

        if self.device == "cuda":
            self.pipe.enable_attention_slicing()

    def generate(self, image: Image.Image, mask: Image.Image, prompt: str, num_samples: int = 4):
        """
        image : RGB PIL image (source)
        mask  : Grayscale PIL image — WHITE = area to inpaint, BLACK = keep original
        prompt: text describing the desired edit
        """
        image = image.resize((512, 512), Image.Resampling.LANCZOS).convert("RGB")
        mask  = mask.resize((512, 512),  Image.Resampling.LANCZOS).convert("L")

        print(f"Inpainting {num_samples} candidates for: \'{prompt}\'")

        result_images = []
        for _ in range(num_samples):
            out = self.pipe(
                prompt=prompt,
                image=image,
                mask_image=mask,
                num_inference_steps=30,
                guidance_scale=7.5,
            ).images[0]
            # Composite back: paste generated onto original outside-mask area
            result_images.append(out)

        return result_images

_pipeline = None
def get_pipeline():
    global _pipeline
    if _pipeline is None:
        _pipeline = InpaintPipeline()
    return _pipeline
"""

with open('model_pipeline.py', 'w') as f:
    f.write(model_pipeline_code)

print('model_pipeline.py created (SD Inpainting).')


In [ ]:
main_code = """from fastapi import FastAPI, HTTPException, Form
from fastapi.middleware.cors import CORSMiddleware
from utils import decode_base64_image, encode_image_base64, preprocess_image
from scoring import get_scorer
from model_pipeline import get_pipeline
import traceback
import base64
import io
import numpy as np
from PIL import Image, ImageFilter

app = FastAPI(title="MagicBrush Pro API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def decode_mask(mask_b64: str, target_size=(512, 512)) -> np.ndarray:
    """Decode base64 mask canvas → float32 numpy array [0.0, 1.0].
    The canvas stores painted strokes in the alpha channel.
    Returns shape (H, W, 1) float32 — 1.0 = edit here, 0.0 = keep original.
    """
    if "," in mask_b64:
        mask_b64 = mask_b64.split(",")[1]
    data    = base64.b64decode(mask_b64)
    img     = Image.open(io.BytesIO(data)).convert("RGBA")
    img     = img.resize(target_size, Image.Resampling.LANCZOS)
    alpha   = np.array(img)[:, :, 3].astype(np.float32) / 255.0
    # Soft threshold: anything painted (alpha > 0.1) counts as mask
    alpha   = (alpha > 0.1).astype(np.float32)
    # Slight dilation to cover brush edge antialiasing
    pil_a   = Image.fromarray((alpha * 255).astype(np.uint8), mode="L")
    pil_a   = pil_a.filter(ImageFilter.MaxFilter(5))
    alpha   = np.array(pil_a).astype(np.float32) / 255.0
    return alpha[:, :, np.newaxis]   # (H, W, 1)

def hard_composite(original, generated, mask_alpha):
    """
    Pixel-perfect compositing:  result = generated * mask + original * (1 - mask)
    original, generated : PIL RGB images at 512x512
    mask_alpha          : numpy float32 (H, W, 1)  values 0..1
    """
    orig_arr = np.array(original.resize((512,512)).convert("RGB")).astype(np.float32)
    gen_arr  = np.array(generated.resize((512,512)).convert("RGB")).astype(np.float32)
    blended  = gen_arr * mask_alpha + orig_arr * (1.0 - mask_alpha)
    return Image.fromarray(blended.clip(0, 255).astype(np.uint8))

def make_full_mask_alpha(size=(512, 512)):
    return np.ones((*size, 1), dtype=np.float32)

@app.on_event("startup")
async def startup_event():
    print("Initializing models...")
    get_pipeline()
    get_scorer()
    print("Application startup complete.")

@app.post("/generate")
async def generate_images(
    image:  str = Form(...),
    prompt: str = Form(...),
    mask:   str = Form(None),
):
    try:
        orig_img = decode_base64_image(image)
        proc_img = preprocess_image(orig_img)   # 512x512 RGB PIL

        # --- Decode mask ---
        if mask and len(mask) > 100:
            print("Mask received — decoding painted region...")
            mask_alpha = decode_mask(mask, target_size=(512, 512))
            # Convert to PIL L-mode for SD pipeline (white=inpaint, black=keep)
            mask_pil   = Image.fromarray((mask_alpha[:,:,0] * 255).astype(np.uint8), mode="L")
        else:
            print("No mask — inpainting full image.")
            mask_alpha = make_full_mask_alpha()
            mask_pil   = Image.new("L", (512, 512), 255)

        # --- Run SD Inpainting ---
        pipeline = get_pipeline()
        generated_images = pipeline.generate(proc_img, mask_pil, prompt, num_samples=4)

        # --- HARD COMPOSITE: enforce exact painted boundary ---
        # SD Inpainting bleeds slightly outside the mask. This step ensures
        # ONLY pixels inside the painted brush strokes are replaced.
        composited_images = [hard_composite(proc_img, g, mask_alpha) for g in generated_images]

        # --- Score & rank ---
        scorer  = get_scorer()
        results = []
        for g_img in composited_images:
            scores  = scorer.score(proc_img, g_img, prompt)
            img_b64 = encode_image_base64(g_img)
            results.append({"image": img_b64, "scores": scores})

        results.sort(key=lambda x: x["scores"]["final_score"], reverse=True)
        return {"samples": results}

    except Exception as e:
        print("Error during generation:")
        print(traceback.format_exc())
        raise HTTPException(status_code=500, detail=str(e))
"""

with open('main.py', 'w') as f:
    f.write(main_code)

print('main.py created with hard pixel-level mask compositing.')


In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
import main

# -----------------------------------------------------
# PLACE YOUR NGROK AUTH TOKEN HERE (Free at ngrok.com)
# -----------------------------------------------------
!ngrok config add-authtoken YOUR_NGROK_TOKEN_HERE

# Fix for asyncio event loop in Jupyter notebooks
nest_asyncio.apply()

port = 8000

# Disconnect previous ngrok tunnels if any exist
ngrok.kill()

public_url = ngrok.connect(port).public_url
print('\n=============================================================')
print('âœ¨ MAGICBRUSH PRO GPU BACKEND IS BOOTING UP âœ¨')
print('=============================================================')
print(f'\n->  PASTE THIS URL INTO YOUR LOCAL app.js:  {public_url}  <-')
print('\n=============================================================\n')

# Start the FastAPI server (This will block the cell from finishing, which is correct)
uvicorn.run(main.app, host='0.0.0.0', port=port)
